# 02. Сбор городских слоёв Малой Охты

Слои по bbox v3 (вся Охта + запас ~250 м) из PBF-выписки СЗФО (Geofabrik, 02.11.2026):

1. `scripts/extract_okhta.py` (pyosmium, без Overpass — pyrosm с bbox терял объекты) → `data/processed/{building,roads,green,water,services,stations}.geojson`;
2. ОКН: фильтр ЕГРОКН по границам + ручная привязка Ниеншанца;
3. Возраст домов: вырезка HOITH по bbox (ogr2ogr);
4. Сводки по версиям границ.

Все внешние запросы закэшированы/зафиксированы в `docs/data_catalog.md` — повторные прогоны работают офлайн и не меняют цифры.

In [15]:
# Настройка
import json
import time
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import shapely
from shapely.geometry import shape, LineString, Point, Polygon
from shapely.ops import nearest_points

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RAW, PROC, CACHE = PROJECT / "data/raw", PROJECT / "data/processed", PROJECT / "data/raw/cache"
CRS_METRIC = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson")
v3 = versions.loc[versions.version == "v3", "geometry"].iloc[0]
# bbox v3 с запасом 200 м
minx, miny, maxx, maxy = v3.bounds
db = 200 / 74000  # ~200 м в градусах по широте СПб
BB = (miny - db, minx - db, maxy + db, maxx + db)  # south, west, north, east (порядок Overpass)
print("bbox запросов:", [round(v, 4) for v in BB])

OVERPASS = "https://overpass.kumi.systems/api/interpreter"

def overpass(query, cache_name, refresh=False):
    """Запрос к Overpass с кэшем в CACHE/cache_name (зеркало kumi.systems надёжнее основного)."""
    path = CACHE / cache_name
    if path.exists() and not refresh:
        return json.load(open(path, encoding="utf-8"))
    r = requests.post(OVERPASS, data={"data": query}, timeout=300,
                      headers={"User-Agent": "malaya-okhta-strategy-academic/0.1"})
    data = r.json()
    json.dump(data, open(path, "w"), ensure_ascii=False)
    return data

bb = ",".join(f"{v:.4f}" for v in BB)
print("готово")

bbox запросов: [59.9127, 30.3916, 59.9632, 30.4513]
готово


In [16]:
# Слои OSM выгружает scripts/extract_okhta.py (pyosmium, PBF СЗФО от Geofabrik — см. docs/data_catalog.md).
# pyrosm с bounding_box терял объекты (2 780 зданий вместо 3 130), поэтому выгрузка отдельным скриптом.
import subprocess
import sys

result = subprocess.run([sys.executable, str(PROJECT / "scripts/extract_okhta.py")],
                        capture_output=True, text=True)
print(result.stdout[-1500:])
assert result.returncode == 0, result.stderr[-800:]

проход 1: ways, узлы, relations рек (FileProcessor + locations)...
  зданий 3130, дорог 11370, зелени 1642, воды(ways) 7, сервисов 231, станций 4
  мультиполигонов рек: 1
проход 2: 490 member-ways рек...
  полигонов из мультиполигонов: 1

итоги:
  building: 3130
  roads: 11370
  green: 1642
  water: 8 (полигоны)
  services: 231
  stations: 4



In [17]:
# Загрузка готовых слоёв
building = gpd.read_file(PROC / "building.geojson")
building["area_m2"] = building.to_crs(CRS_METRIC).geometry.area
roads_g = gpd.read_file(PROC / "roads.geojson")
green_g = gpd.read_file(PROC / "green.geojson")
water = gpd.read_file(PROC / "water.geojson")
pois_g = gpd.read_file(PROC / "services.geojson")
rail_g = gpd.read_file(PROC / "stations.geojson")
print("зданий:", len(building), "| застройка, га:", round(building.area_m2.sum() / 1e4))
print("дорог:", len(roads_g), "| зелени:", len(green_g), "| воды:", len(water),
      "| сервисов:", len(pois_g), "| станций:", len(rail_g))
print(building.building.value_counts().head(8).to_dict())

зданий: 3130 | застройка, га: 308
дорог: 11370 | зелени: 1642 | воды: 8 | сервисов: 231 | станций: 4
{'yes': 1335, 'apartments': 711, 'service': 239, 'garages': 180, 'industrial': 163, 'office': 95, 'roof': 65, 'kindergarten': 52}


In [18]:
# Сводка по версиям границ: застройка, сервисы, станции
for _, row in versions.iterrows():
    inside = building.within(row.geometry)
    n_s = int(pois_g.within(row.geometry).sum())
    a = building[inside].area_m2.sum() / 1e4
    print(f"{row['version']}: зданий {int(inside.sum())} ({a:.0f} га), сервисов {n_s}")

print("\nстанции:", rail_g["name"].tolist() if "name" in rail_g else rail_g.shape[0])
print("\nсервисы по типам:")
print(pois_g["amenity"].value_counts().head(12).to_dict())

v1: зданий 217 (16 га), сервисов 30
v2: зданий 694 (62 га), сервисов 95
v3: зданий 1674 (153 га), сервисов 160
v4: зданий 1578 (143 га), сервисов 159

станции: ['Новочеркасская', 'Ладожская', 'Санкт-Петербург-Ладожский', 'Полюстрово']

сервисы по типам:
{'pharmacy': 79, 'bank': 36, 'doctors': 24, 'clinic': 23, 'library': 13, 'post_office': 12, 'police': 11, 'kindergarten': 10, 'place_of_worship': 8, 'cinema': 4, 'school': 3, 'community_centre': 2}


In [19]:
# ОКН: фильтр ЕГРОКН по версиям + ручная привязка Ниеншанца (в ЕГРОКН нет ни адреса, ни координат)
MANUAL = {  # по образцу Коломны: объекты без координат, привязка по известному месту
    "Ниеншанц": {"lon": 30.4106, "lat": 59.9461, "note": "Охтинский мыс, раскоп «Охта-1» у Красногвардейской пл."},
}

okn = pd.read_csv(RAW / "egrokn_spb_all.csv", encoding="utf-8-sig")
okn_rows = okn.dropna(subset=["lon", "lat"]).copy()
okn_rows["note"] = ""

# добавляем вручную привязанные
for name, meta in MANUAL.items():
    src = okn[okn["name"].str.startswith(name, na=False)]
    for _, r in src.iterrows():
        okn_rows = pd.concat([okn_rows, pd.DataFrame([{**r.to_dict(), "lon": meta["lon"],
                                                        "lat": meta["lat"], "note": meta["note"]}])],
                              ignore_index=True)

okn_g = gpd.GeoDataFrame(okn_rows, geometry=gpd.points_from_xy(okn_rows["lon"], okn_rows["lat"]),
                          crs="EPSG:4326")
# оставляем только объекты в v3
okn_g = okn_g[okn_g.within(versions.loc[versions.version == "v3", "geometry"].iloc[0])].copy()
okn_g.to_file(PROC / "okn.geojson", driver="GeoJSON")
print("ОКН в v3 (с ручной привязкой):", len(okn_g))
print(okn_g["cat"].value_counts().to_dict())
print("\nфедеральные:")
for _, r in okn_g[okn_g.cat == "Федерального значения"][["name", "address"]].iterrows():
    print("  ", r["name"][:70], "|", str(r["address"])[:40])

ОКН в v3 (с ручной привязкой): 60
{'Регионального значения': 43, 'Федерального значения': 17}

федеральные:
   Могила Фомина Н.П. (1864-1943), композитора, пианиста | Санкт-Петербург, Металлистов просп., 5, 
   Могила Турнера Г.И. (1858-1941), хирурга-ортопеда | Санкт-Петербург, Металлистов просп., 5, 
   Могила Поленова А.Л. (1871-1947), нейрохирурга, травматолога | Санкт-Петербург, Металлистов просп., 5, 
   Могила Малиновского В.Ф. (1765-1814), публициста, просветителя, первог | Санкт-Петербург, Металлистов просп., 5, 
   Могила Крупчатникова М.Я. (1897-1947), ученого-артиллериста, Героя Соц | Санкт-Петербург, Металлистов просп., 5, 
   Могила Деммени Е.С. (1898-1969), основателя Ленинградского театра куко | Санкт-Петербург, Металлистов просп., 5, 
   Могила Графтио Г.О. (1869-1949), гидроэнергетика | Санкт-Петербург, Металлистов просп., 5, 
   Могила Велланского (Кавунника) Д.М. (1774-1847), медика, философа | Санкт-Петербург, Металлистов просп., 5, 
   Могила Буяльского И.В. (1789

In [20]:
# Возраст домов: вырезка из «Карты возраста домов» (HOITH) по bbox через ogr2ogr.
# Исходник (1,8 ГБ) вне репозитория: ~/Workspace/Городская идентичность/HOITH_database_231118.gpkg
# Схема: r_year_int (год), r_name, r_adress, r_floors, isApartmentBuilding; CRS метрическая внутри gpkg.
import subprocess

HOITH = Path("/home/mfthesun/Workspace/Городская идентичность/HOITH_database_231118.gpkg")
hoith_cut = RAW / "hoith_okhta.gpkg"
if HOITH.exists() and not hoith_cut.exists():
    subprocess.run(["ogr2ogr", "-f", "GPKG", str(hoith_cut), str(HOITH),
                    "all_cities_181123", "-spat", str(minx), str(miny), str(maxx), str(maxy),
                    "-spat_srs", "EPSG:4326"], check=True)

if hoith_cut.exists():
    hoith = gpd.read_file(hoith_cut).to_crs(4326)
    hoith = hoith[hoith.intersects(v3)].copy()
    hoith = hoith.reset_index().rename(columns={"index": "hoith_id"})
    hoith["year"] = pd.to_numeric(hoith.get("r_year_int"), errors="coerce")
    cols = ["hoith_id", "year", "r_name", "r_adress", "r_floors", "isApartmentBuilding", "geometry"]
    hoith[[c for c in cols if c in hoith.columns]].to_file(PROC / "house_age.geojson", driver="GeoJSON")
    print("домов с возрастом в v3:", len(hoith), "| с известным годом:", int(hoith.year.notna().sum()))
    bins = pd.cut(hoith.year, [1700, 1917, 1946, 1961, 1981, 1992, 2006, 2027],
                  labels=["до 1917", "1917–1945", "1946–1960", "1961–1980", "1981–1991", "1992–2005", "2006+"],
                  right=False)
    print(bins.value_counts().sort_index().to_dict())
else:
    print("HOITH недоступен — слой возраста домов пропускается")

домов с возрастом в v3: 1596 | с известным годом: 1095
{'до 1917': 57, '1917–1945': 90, '1946–1960': 264, '1961–1980': 471, '1981–1991': 63, '1992–2005': 42, '2006+': 108}
